In [ ]:
import os

os.chdir("..")

In [ ]:
from fsdc_calories import DataCal
import polars as pl
import importlib.resources as resources
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px


dc = DataCal()

In [ ]:
import plotly.graph_objects as go
import polars as pl
from importlib import resources

# Assuming dc is your data processing module/class imported beforehand
df = dc.process_price(agriculture_filter=True)
path_data = str(
    resources.files("fsdc_calories").joinpath("data/hts4_walk.parquet")
)

desc_df = pl.read_parquet(path_data).rename(
    {"code_4": "hs4", "Description": "hts_desc"}
)
df = df.join(desc_df, on="hs4", how="inner", validate="m:1")

# 1. Get the last 4 unique dates (quarters) sorted chronologically
all_dates = df.select("date").unique().sort("date")["date"].to_list()
last_4_dates = all_dates[-4:] if len(all_dates) >= 4 else all_dates

fig = go.Figure()

# Enhanced hover template
hover_template = (
    "<b>Description:</b> %{y}<br>"
    "<b>HS4 Code:</b> %{customdata[0]}<br>"
    "<b>Price Imports YoY:</b> %{customdata[1]:.1f}%<br>"
    "<extra></extra>"
)

trace_counter = 0
date_trace_indices = {} 

for d_idx, target_date in enumerate(last_4_dates):
    # Filter data and shorten descriptions further for mobile screens (3 words max)
    quarter_data = (
        df.filter(pl.col("date") == target_date)
        .with_columns(
            actual_yoy=pl.col("price_imports_yoy"),
            plot_yoy=pl.col("price_imports_yoy").clip(-100, 100),
            text_label=(
                pl.col("price_imports_yoy").round(1).cast(pl.Utf8) + "%"
            ).str.replace(r"^([0-9])", r"+$1"),
            short_desc=(
                pl.col("hts_desc").str.split(" ").list.head(3).list.join(" ")
                + pl.when(pl.col("hts_desc").str.split(" ").list.len() > 3)
                .then(pl.lit("..."))
                .otherwise(pl.lit(""))
            ),
        )
        .with_columns(
            short_desc=pl.when(
                pl.col("short_desc").is_duplicated().over("short_desc")
            )
            .then(
                pl.col("short_desc")
                + " ("
                + (pl.cum_count("short_desc").over("short_desc").cast(pl.Utf8))
                + ")"
            )
            .otherwise(pl.col("short_desc"))
        )
    )

    top_data = (
        quarter_data.sort("actual_yoy", descending=True)
        .head(30)
        .to_pandas()
        .iloc[::-1]
    )

    bottom_data = (
        quarter_data.sort("actual_yoy", descending=False)
        .head(50)
        .to_pandas()
        .iloc[::-1]
    )

    # Top Trace
    fig.add_trace(
        go.Bar(
            x=top_data["plot_yoy"],
            y=top_data["short_desc"],
            text=top_data["text_label"],
            textposition="outside",
            orientation="h",
            customdata=list(zip(top_data["hs4"], top_data["actual_yoy"])),
            hovertemplate=hover_template,
            marker=dict(
                color=top_data["plot_yoy"],
                colorscale="Viridis",
                showscale=False,
                opacity=0.9,
            ),
            visible=(d_idx == len(last_4_dates) - 1), 
        )
    )
    top_idx = trace_counter
    trace_counter += 1

    # Bottom Trace
    fig.add_trace(
        go.Bar(
            x=bottom_data["plot_yoy"],
            y=bottom_data["short_desc"],
            text=bottom_data["text_label"],
            textposition="outside",
            orientation="h",
            customdata=list(zip(bottom_data["hs4"], bottom_data["actual_yoy"])),
            hovertemplate=hover_template,
            marker=dict(
                color=bottom_data["plot_yoy"],
                colorscale="Viridis_r",
                showscale=False,
                opacity=0.9,
            ),
            visible=False,
        )
    )
    bottom_idx = trace_counter
    trace_counter += 1

    date_trace_indices[target_date] = {"top": top_idx, "bottom": bottom_idx}

total_traces = trace_counter
top_btn_visibility = [False] * total_traces
bottom_btn_visibility = [False] * total_traces

latest_date = last_4_dates[-1]
top_btn_visibility[date_trace_indices[latest_date]["top"]] = True
bottom_btn_visibility[date_trace_indices[latest_date]["bottom"]] = True

metric_buttons = [
    dict(
        label="Top 50 Categories",
        method="update",
        args=[
            {"visible": top_btn_visibility},
            {"xaxis.range": [0, 120], "xaxis.title_text": "YoY % Change"},
        ],
    ),
    dict(
        label="Bottom 50 Categories",
        method="update",
        args=[
            {"visible": bottom_btn_visibility},
            {"xaxis.range": [-120, 0], "xaxis.title_text": "YoY % Change"},
        ],
    ),
]

quarter_buttons = []
for target_date in last_4_dates:
    date_str = str(target_date)
    q_top_vis = [False] * total_traces
    q_bot_vis = [False] * total_traces
    
    q_top_vis[date_trace_indices[target_date]["top"]] = True
    q_bot_vis[date_trace_indices[target_date]["bottom"]] = True

    quarter_buttons.append(
        dict(
            label=f"Quarter: {date_str}",
            method="update",
            args=[
                {"visible": q_top_vis}, 
                {"xaxis.range": [0, 120], "xaxis.title_text": "YoY % Change"},
            ],
        )
    )

# Mobile-Optimized Layout Settings
fig.update_layout(
    updatemenus=[
        # Dropdown 1: Stacked cleanly at the top-left
        dict(
            type="dropdown",
            direction="down",
            x=0.0,
            xanchor="left",
            y=1.35,
            yanchor="top",
            bgcolor="#F8F9F9",
            bordercolor="#BDC3C7",
            borderwidth=1,
            font=dict(size=10, color="#2C3E50"),
            buttons=metric_buttons,
        ),
        # Dropdown 2: Stacked directly below the first dropdown for mobile viewports
        dict(
            type="dropdown",
            direction="down",
            x=0.0,
            xanchor="left",
            y=1.20,
            yanchor="top",
            bgcolor="#F8F9F9",
            bordercolor="#BDC3C7",
            borderwidth=1,
            font=dict(size=10, color="#2C3E50"),
            buttons=quarter_buttons,
        ),
    ],
    template="plotly_white",
    title=dict(
        text="<b>Import Price Volatility</b><br><sup>YoY Changes by Category</sup>",
        font=dict(size=14, color="#2C3E50"),
        x=0.5,
        xanchor="center",
        y=0.98,
        yanchor="top",
    ),
    font=dict(size=10, color="#34495E"),
    height=850,  # Reduced height to fit smaller screens better
    margin=dict(l=100, r=50, t=240, b=50), # Adjusted margins for mobile constraints
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)",
)

fig.update_xaxes(
    title_text="YoY % Change",
    range=[0, 120],
    showgrid=True,
    gridcolor="#E5E8E8",
    zeroline=True,
    zerolinecolor="#2C3E50",
)

fig.update_yaxes(showticklabels=True, tickfont=dict(size=8), showgrid=False)

# Enable responsive scaling for web/mobile embedding framework containers
fig.show(config={"responsive": True, "displayModeBar": False})

In [ ]:
# Replace fig.show(...) with this:
fig.write_html("mobile_test.html", config={"responsive": True, "displayModeBar": False})

In [ ]:
python -m http.server 8001